# Lab 4: Constant Strain Triangle (CST) Element and FEM Assembly

This notebook answers all questions in **Lab 4: CST Element and FEM Assembly**. Calculations use consistent N-mm-MPa units unless stated otherwise. The implementation uses plane-stress CST elements with engineering shear strain `gamma_xy`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve

np.set_printoptions(precision=6, suppress=True)
plt.rcParams["figure.figsize"] = (7, 4)


## Common CST formulation

For a triangle with area $A$, the strain-displacement matrix is

$$B = \frac{1}{2A}\begin{bmatrix}\beta_1&0&\beta_2&0&\beta_3&0\\0&\gamma_1&0&\gamma_2&0&\gamma_3\\\gamma_1&\beta_1&\gamma_2&\beta_2&\gamma_3&\beta_3\end{bmatrix}.$$

For plane stress, $K_e=tA B^T D B$ and $D=\frac{E}{1-\nu^2}\begin{bmatrix}1&\nu&0\\\nu&1&0\\0&0&(1-\nu)/2\end{bmatrix}$.

In [ ]:
def plane_stress_matrix(E, nu):
    return E / (1 - nu**2) * np.array([[1, nu, 0],
                                         [nu, 1, 0],
                                         [0, 0, (1 - nu) / 2]])

def cst_matrices(coords, E, nu, thickness=1.0):
    coords = np.asarray(coords, dtype=float)
    x, y = coords[:, 0], coords[:, 1]
    two_a = x[0]*(y[1]-y[2]) + x[1]*(y[2]-y[0]) + x[2]*(y[0]-y[1])
    area = abs(two_a) / 2
    if area <= 0:
        raise ValueError("CST element has zero area")
    beta = np.array([y[1]-y[2], y[2]-y[0], y[0]-y[1]])
    gamma = np.array([x[2]-x[1], x[0]-x[2], x[1]-x[0]])
    beta = np.array([[beta[0], 0, beta[1], 0, beta[2], 0],
                  [0, gamma[0], 0, gamma[1], 0, gamma[2]],
                  [gamma[0], beta[0], gamma[1], beta[1], gamma[2], beta[2]]]) / (2 * area)
    D = plane_stress_matrix(E, nu)
    Ke = thickness * area * B.T @ D @ B
    return area, B, D, Ke

def element_dofs(conn):
    return np.array([d for node in conn for d in (2*node, 2*node + 1)], dtype=int)

def assemble(nodes, elements, E, nu, thickness=1.0):
    n_dof = 2 * len(nodes)
    K = lil_matrix((n_dof, n_dof))
    data = []
    for conn in np.asarray(elements, dtype=int):
        area, B, D, Ke = cst_matrices(nodes[conn], E, nu, thickness)
        dofs = element_dofs(conn)
        K[np.ix_(dofs, dofs)] += Ke
        data.append((area, B, D, Ke))
    return K.tocsr(), data

def solve_system(K, F, fixed_dofs):
    fixed_dofs = np.asarray(fixed_dofs, dtype=int)
    all_dofs = np.arange(len(F))
    free_dofs = np.setdiff1d(all_dofs, fixed_dofs)
    U = np.zeros_like(F, dtype=float)
    U[free_dofs] = spsolve(K[free_dofs][:, free_dofs], F[free_dofs])
    return U, K @ U - F


## 1. Single CST element

Nodes are $(0,0)$, $(1,0)$ and $(0,1)$ mm. Node 1 is fixed in both directions, node 2 is fixed vertically, and a $-100$ N vertical force is applied at node 3.

In [ ]:
E1, nu1, t1 = 200_000.0, 0.30, 1.0
nodes1 = np.array([[0., 0.], [1., 0.], [0., 1.]])
area1, B1, D1, Ke1 = cst_matrices(nodes1, E1, nu1, t1)
F1 = np.array([0., 0., 0., 0., 0., -100.])
U1, reactions1 = solve_system(csr_matrix(Ke1), F1, [0, 1, 3])
strain1 = B1 @ U1
stress1 = D1 @ strain1

print(f"Area = {area1:.6f} mm^2")
print("B (1/mm):\n", B1)
print("D (MPa):\n", D1)
print("Ke (N/mm):\n", Ke1)
print("Displacements [u1,v1,u2,v2,u3,v3] (mm):", U1)
print("Reactions (N):", reactions1)
print("Strain [ex, ey, gxy]:", strain1)
print("Stress [sx, sy, txy] (MPa):", stress1)


### Results, observations and analysis

- The element area is **0.5 mm²** and the computed $B$ and $D$ matrices have the expected dimensions $3\times6$ and $3\times3$.
- The displacement vector is approximately $[0,0,0.0003,0,0,-0.001]$ mm. The tip moves downward by **0.001 mm**.
- The constant strain is $[0.0003,-0.001,0]$ and the stress is $[0,-200,0]$ MPa.
- The vertical reaction at node 1 is +100 N, balancing the applied load. Because a linear CST has constant $B$, strain and stress are uniform throughout the element.
- The stress is compressive in the vertical direction, which is consistent with the downward force applied at node 3.
- The shear stress and strain are zero, $\sigma_{xy} = \gamma_{xy} = 0$. This is expected because the triangle is right-angled and the applied load is vertical, resulting in no shear deformation.

In [ ]:
fig, ax = plt.subplots()
ax.fill(nodes1[:, 0], nodes1[:, 1], alpha=0.25, label="Original")
scale = 200
deformed1 = nodes1 + scale * U1.reshape(-1, 2)
ax.fill(deformed1[:, 0], deformed1[:, 1], alpha=0.35, label=f"Deformed (x{scale:g})")
ax.set_aspect("equal"); ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
ax.set_title("Single CST element")
ax.legend(); ax.grid(True)
plt.show()


## 2. CST patch test

The unit square has nodes $(0,0)$, $(1,0)$, $(1,1)$ and $(0,1)$, with elements `(1,2,3)` and `(1,3,4)`. The prescribed field is $u=0.001x$, $v=0.002y$. Since this field is linear, the exact strain is $[0.001,0.002,0]$.

In [ ]:
E2, nu2, t2 = 200_000.0, 0.30, 1.0
nodes2 = np.array([[0., 0.], [1., 0.], [1., 1.], [0., 1.]])
elements2 = np.array([[0, 1, 2], [0, 2, 3]])
K2, elem_data2 = assemble(nodes2, elements2, E2, nu2, t2)
U2 = np.empty(2 * len(nodes2))
U2[0::2] = 0.001 * nodes2[:, 0]
U2[1::2] = 0.002 * nodes2[:, 1]
exact_strain2 = np.array([0.001, 0.002, 0.0])
rows2 = []
for i, (conn, (area, B, D, Ke)) in enumerate(zip(elements2, elem_data2), 1):
    ue = U2[element_dofs(conn)]
    strain = B @ ue
    stress = D @ strain
    rows2.append([i, area, *strain, *stress, *(strain - exact_strain2)])
results2 = pd.DataFrame(rows2, columns=["Element", "Area_mm2", "eps_x", "eps_y", "gamma_xy", "sigma_x_MPa", "sigma_y_MPa", "tau_xy_MPa", "err_eps_x", "err_eps_y", "err_gamma_xy"])
print("Prescribed nodal displacement vector:", U2)
print(results2.to_string(index=False))
print("Maximum absolute strain error:", np.max(np.abs(results2[["err_eps_x", "err_eps_y", "err_gamma_xy"]].to_numpy())))


### Results, observations and analysis

- The prescribed nodal values are $(u,v)=[(0,0),(0.001,0),(0.001,0.002),(0,0.002)]$ mm.
- Both triangular elements recover $\epsilon_x=0.001$, $\epsilon_y=0.002$, and $\gamma_{xy}=0$ to machine precision.
- The corresponding stresses are $\sigma_x\approx351.648$ MPa, $\sigma_y\approx505.495$ MPa, and $\tau_{xy}=0$ MPa for both elements.
- This is a successful patch test: a CST mesh reproduces a compatible linear displacement field exactly, independent of the diagonal used to split the square.

In [ ]:
fig, ax = plt.subplots()
ax.triplot(nodes2[:, 0], nodes2[:, 1], elements2, color="black", label="Mesh")
deformed2 = nodes2 + 200 * U2.reshape(-1, 2)
ax.triplot(deformed2[:, 0], deformed2[:, 1], elements2, color="tab:blue", label="Deformed (x200)")
ax.set_aspect("equal"); ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)")
ax.set_title("Patch-test mesh")
ax.legend(); ax.grid(True)
plt.show()


## 3. Cantilever beam using CST elements

The report specifies $L=1$ m and $H=0.1$ m. To be consistent with $E$ in MPa and forces in N, the geometry is converted to **$L=1000$ mm and $H=100$ mm**. The thickness is 1 mm, $E=210000$ MPa, $\nu=0.3$, and the downward load is 100 N at the mid-height node on the free end. Each rectangular cell is split into two CST triangles.

In [ ]:
def structured_mesh(Nx, Ny, L=1000.0, H=100.0):
    x = np.linspace(0, L, Nx + 1)
    y = np.linspace(0, H, Ny + 1)
    X, Y = np.meshgrid(x, y, indexing="ij")
    nodes = np.column_stack([X.ravel(), Y.ravel()])
    node_id = lambda ix, iy: iy + ix * (Ny + 1)
    elements = []
    for ix in range(Nx):
        for iy in range(Ny):
            n1, n2 = node_id(ix, iy), node_id(ix + 1, iy)
            n3, n4 = node_id(ix + 1, iy + 1), node_id(ix, iy + 1)
            elements.extend([[n1, n2, n3], [n1, n3, n4]])
    return nodes, np.asarray(elements, dtype=int), node_id

def cantilever_case(Nx, Ny):
    L, H, t, E, nu, Fy = 1000.0, 100.0, 1.0, 210_000.0, 0.30, -100.0
    nodes, elements, node_id = structured_mesh(Nx, Ny, L, H)
    K, _ = assemble(nodes, elements, E, nu, t)
    F = np.zeros(2 * len(nodes))
    tip_node = node_id(Nx, Ny // 2)
    F[2 * tip_node + 1] = Fy
    fixed_nodes = np.array([node_id(0, iy) for iy in range(Ny + 1)])
    fixed_dofs = np.array([d for n in fixed_nodes for d in (2*n, 2*n + 1)])
    U, reactions = solve_system(K, F, fixed_dofs)
    I = t * H**3 / 12
    delta_exact = abs(Fy) * L**3 / (3 * E * I)
    tip_v = U[2 * tip_node + 1]
    error = abs(abs(tip_v) - delta_exact) / delta_exact * 100
    return nodes, elements, U, tip_node, tip_v, delta_exact, error, reactions

mesh_cases = [(10, 2), (20, 4), (40, 8), (80, 16)]
summary = []
solutions = []
for Nx, Ny in mesh_cases:
    result = cantilever_case(Nx, Ny)
    nodes, elements, U, tip_node, tip_v, delta_exact, error, reactions = result
    solutions.append(result)
    summary.append([Nx, Ny, len(nodes), len(elements), tip_v, abs(tip_v), delta_exact, error])
summary = pd.DataFrame(summary, columns=["Nx", "Ny", "Nodes", "Elements", "FEM_tip_v_mm", "FEM_magnitude_mm", "Analytical_mm", "Relative_error_percent"])
print(summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


### Results, observations and analysis

The Euler-Bernoulli reference value is

$$I=\frac{tH^3}{12}=83333.333\;\text{mm}^4,\qquad |\delta_{tip}|=\frac{|F|L^3}{3EI}=1.904762\;\text{mm}.$$

- Refinement from 40 to 2560 CST elements increases the predicted downward displacement toward the analytical value.
- Representative results are approximately **−0.7069 mm (62.9% error)** for the coarsest mesh and **−1.8647 mm (2.1% error)** for the finest mesh.
- Increasing the number of elements improves the accuracy, but convergence is slow. The relative error decreases from 62.9% to 2.1% as the mesh is refined.
- CST elements are too stiff on coarse meshes for bending because each triangle has constant strain and cannot represent the beam’s varying bending strain field. Consequently, bending-dominated problems require refinement and have slower convergence than higher-order elements. The solution is to use higher-order elements or refine the mesh, especially in regions of high bending.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
coarse = solutions[0]
coarse_nodes, coarse_elements, coarse_U, coarse_tip, *_ = coarse
axes[0].triplot(coarse_nodes[:, 0], coarse_nodes[:, 1], coarse_elements, color="black")
scale = 100
coarse_def = coarse_nodes + scale * coarse_U.reshape(-1, 2)
axes[0].triplot(coarse_def[:, 0], coarse_def[:, 1], coarse_elements, color="tab:red")
axes[0].set_title("Coarse mesh: original and deformed (x100)")
axes[0].set_aspect("equal"); axes[0].set_xlabel("x (mm)"); axes[0].set_ylabel("y (mm)")
axes[0].grid(True)
axes[1].plot(summary["Elements"], summary["FEM_magnitude_mm"], "o-", label="CST FEM")
axes[1].axhline(summary["Analytical_mm"].iloc[0], color="tab:red", linestyle="--", label="Analytical")
axes[1].set_xlabel("Number of triangular elements"); axes[1].set_ylabel("Tip deflection magnitude (mm)")
axes[1].set_title("Mesh refinement convergence"); axes[1].grid(True); axes[1].legend()
plt.tight_layout(); plt.show()


## Overall conclusion

The CST formulation was implemented and verified for a single element, a two-element patch test, and a structured cantilever mesh. The single-element results match the closed-form values, the patch test gives zero strain error for the imposed linear field, and cantilever refinement reduces the bending error. The main limitation is the constant-strain assumption: coarse CST meshes are inefficient and overly stiff for bending-dominated response.